In [ ]:
import random, json
from collections import Counter

# ── CONFIGURAÇÕES ──────────────────────────────────────────────

SEXOS = ["masculino", "feminino"]

SYMPTOMS = [
    "dispneia progressiva", "dor torácica opressiva", "febre alta",
    "tosse produtiva", "confusão mental", "cefaleia súbita intensa",
    "fadiga", "edema em membros inferiores", "náuseas", "palpitações",
    "síncope", "hemoptise", "perda de peso", "dor abdominal",
    "rigidez de nuca", "vômitos", "diarreia", "dor lombar",
    "dispneia súbita", "febre baixa"
]

DISEASE_SYMPTOM_MAP = {
    "insuficiência cardíaca descompensada": ["dispneia progressiva", "edema em membros inferiores", "fadiga", "palpitações"],
    "tromboembolismo pulmonar":             ["hemoptise", "síncope", "dispneia súbita"],
    "pneumonia":                            ["febre alta", "tosse produtiva", "dispneia progressiva", "fadiga"],
    "síndrome coronariana aguda":           ["dor torácica opressiva", "náuseas", "fadiga", "palpitações"],
    "sepse":                                ["febre alta", "confusão mental", "fadiga"],
    "apendicite aguda":                     ["dor abdominal", "náuseas", "vômitos", "febre baixa"],
    "meningite bacteriana":                 ["febre alta", "cefaleia súbita intensa", "rigidez de nuca", "confusão mental"],
    "acidente vascular cerebral":           ["confusão mental", "cefaleia súbita intensa"],
    "cólica renal":                         ["dor lombar", "náuseas", "vômitos"],
    "gastroenterite aguda":                 ["náuseas", "vômitos", "diarreia", "dor abdominal"],
    "hemorragia subaracnoide":              ["cefaleia súbita intensa", "confusão mental", "vômitos"],
}

EXAMS_BY_DISEASE = {
    "insuficiência cardíaca descompensada": ["ecocardiograma", "BNP", "radiografia de tórax"],
    "tromboembolismo pulmonar":             ["angiotomografia pulmonar", "d-dímero", "gasometria arterial"],
    "pneumonia":                            ["radiografia de tórax", "hemograma", "proteína C reativa"],
    "síndrome coronariana aguda":           ["ECG", "troponina", "ecocardiograma"],
    "sepse":                                ["hemograma", "lactato", "hemocultura"],
    "apendicite aguda":                     ["hemograma", "proteína C reativa", "ultrassom abdominal"],
    "meningite bacteriana":                 ["punção lombar", "hemograma", "hemocultura"],
    "acidente vascular cerebral":           ["tomografia de crânio", "ressonância magnética", "hemograma"],
    "cólica renal":                         ["ultrassom renal", "urina tipo I", "tomografia abdominal"],
    "gastroenterite aguda":                 ["hemograma", "eletrólitos", "coprocultura"],
    "hemorragia subaracnoide":              ["tomografia de crânio", "punção lombar", "angiografia cerebral"],
}
COMORBIDADES_POOL = [
    "hipertensão arterial", "diabetes tipo 2", "insuficiência cardíaca",
    "DPOC", "fibrilação atrial", "doença renal crônica",
    "hipotireoidismo", "obesidade", "tabagismo", "etilismo",
    "dislipidemia", "asma", "cirrose hepática", "HIV"
]

QUESTIONS = [
    "Quais diagnósticos devem ser considerados?",
    "Quais exames auxiliariam na investigação?",
    "Qual hipótese diagnóstica é mais provável?",
    "Que exames solicitar inicialmente?",
]
# ── FUNÇÕES ────────────────────────────────────────────────────

def infer_diseases(symptoms):
    scores = {}
    for disease, dsymptoms in DISEASE_SYMPTOM_MAP.items():
        match = len(set(symptoms) & set(dsymptoms))
        if match > 0:
            scores[disease] = match
    return [d for d, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def generate_reasoning(symptoms, comorbidades):
    r = []
    if comorbidades:
        r.append(f"Paciente com histórico de {', '.join(comorbidades)}, o que representa fator de risco relevante para a apresentação atual.")
    if "febre alta" in symptoms or "febre baixa" in symptoms:
        if any(c in comorbidades for c in ["diabetes tipo 2", "doença renal crônica", "HIV"]):
            r.append("Considerando o estado imunossupressor, processo infeccioso deve ser investigado com prioridade.")
        else:
            r.append("A presença de febre sugere processo infeccioso.")
    if "dor torácica opressiva" in symptoms:
        if any(c in comorbidades for c in ["hipertensão arterial", "diabetes tipo 2", "dislipidemia"]):
            r.append("Dor torácica opressiva em paciente com múltiplos fatores de risco cardiovasculares sugere síndrome coronariana aguda como hipótese prioritária.")
        else:
            r.append("Dor torácica opressiva sugere etiologia cardíaca.")
    if "dispneia progressiva" in symptoms:
        r.append("Dispneia progressiva pode indicar comprometimento cardiopulmonar.")
    if "rigidez de nuca" in symptoms and ("febre alta" in symptoms or "cefaleia súbita intensa" in symptoms):
        r.append("Rigidez de nuca associada a febre e alteração do nível de consciência compõe triade de meningite.")
    if "dor abdominal" in symptoms and ("náuseas" in symptoms or "vômitos" in symptoms):
        r.append("Dor abdominal com náuseas/vômitos requer investigação cirúrgica para excluir abdome agudo.")
    if "cefaleia súbita intensa" in symptoms:
        r.append("Cefaleia de início súbito e intensa requer exclusão de hemorragia subaracnoide.")
    if not r:
        r.append("Os sintomas apresentados requerem investigação clínica detalhada.")
    return " ".join(r)

def generate_case():
    idade  = random.randint(18, 90)
    sexo   = random.choice(SEXOS)
    sabrev = "M" if sexo == "masculino" else "F"

    comorbidades = random.sample(COMORBIDADES_POOL, random.randint(1, 3)) if random.random() < 0.6 else []
    symptoms     = random.sample(SYMPTOMS, random.randint(1, 3))
    diseases     = infer_diseases(symptoms) or ["condição clínica a investigar"]
    main  = diseases[0]
    diff  = diseases[1:4]
    exams = EXAMS_BY_DISEASE.get(main, ["hemograma", "função renal"])

    context  = f"Contexto do paciente:\nPaciente {sabrev}, {idade} anos."
    if comorbidades:
        context += f"\nHistórico: {', '.join(comorbidades)}"
    context += f"\n\nSintomas relatados:\n{', '.join(symptoms)}"

    output  = f"Resumo clínico:\nPaciente apresentando {', '.join(symptoms)}."
    if comorbidades:
        output += f" Histórico relevante: {', '.join(comorbidades)}."
    output += f"\n\nRaciocínio clínico:\n{generate_reasoning(symptoms, comorbidades)}"
    output += f"\n\nHipótese diagnóstica principal:\n{main}"
    output += f"\n\nDiagnósticos diferenciais:\n" + "".join(f"- {d}\n" for d in diff)
    output += f"\nExames recomendados:\n" + "".join(f"- {e}\n" for e in exams)

    return context.strip(), random.choice(QUESTIONS), output.strip()

# ── GERAR DATASET ──────────────────────────────────────────────

TOTAL_EXAMPLES = 9000
dataset = []

for _ in range(TOTAL_EXAMPLES):
    context, question, answer = generate_case()
    dataset.append({"messages": [
        {"role": "system",    "content": "Você é um assistente médico clínico que analisa sintomas e sugere hipóteses diagnósticas e exames."},
        {"role": "user",      "content": context + "\n\nPergunta:\n" + question},
        {"role": "assistant", "content": answer}
    ]})

with open("dataset_clinical_cases_generated.jsonl", "w", encoding="utf8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# ── VALIDAR ────────────────────────────────────────────────────

hipoteses = []
for row in dataset:
    for msg in row["messages"]:
        if msg["role"] == "assistant":
            for line in msg["content"].split("\n"):
                if line.strip() and not any(line.startswith(p) for p in
                        ["Resumo", "Raciocínio", "Hipótese", "Diagnósticos", "Exames", "-", "Histórico", "Paciente"]):
                    hipoteses.append(line.strip())
                    break

print(f"Dataset gerado: {len(dataset)} casos\n")
print("Distribuicao de doencas:")
for doenca, count in Counter(hipoteses).most_common(15):
    print(f"  {doenca:<45} {count:>5}  ({count/TOTAL_EXAMPLES*100:.1f}%)")

com_hist = sum(1 for row in dataset if "Histórico:" in row["messages"][1]["content"])
print(f"\nCasos com comorbidades: {com_hist}/{TOTAL_EXAMPLES} ({com_hist/TOTAL_EXAMPLES*100:.1f}%)")

Dataset gerado: 9000 casos

Distribuicao de doencas:
  Os sintomas apresentados requerem investigação clínica detalhada.  2019  (22.4%)
  insuficiência cardíaca descompensada           1545  (17.2%)
  tromboembolismo pulmonar                       1035  (11.5%)
  pneumonia                                       605  (6.7%)
  apendicite aguda                                584  (6.5%)
  síndrome coronariana aguda                      506  (5.6%)
  A presença de febre sugere processo infeccioso.   498  (5.5%)
  meningite bacteriana                            369  (4.1%)
  Dor torácica opressiva sugere etiologia cardíaca.   287  (3.2%)
  Dispneia progressiva pode indicar comprometimento cardiopulmonar.   254  (2.8%)
  Cefaleia de início súbito e intensa requer exclusão de hemorragia subaracnoide.   248  (2.8%)
  sepse                                           211  (2.3%)
  gastroenterite aguda                            208  (2.3%)
  cólica renal                                    193  (2.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

pasta = "/content/drive/MyDrive/FIAP-TC-Fase3"
os.makedirs(pasta, exist_ok=True)

!cp "dataset_clinical_cases_generated.jsonl" "{pasta}/"
print(f"Salvo em: {pasta}/dataset_clinical_cases_generated.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Salvo em: /content/drive/MyDrive/FIAP-TC-Fase3/dataset_clinical_cases_generated.jsonl
